# 솔챗 전체를 Colab에서 실행하기 (모델 직접 로드 + 웹 서버 + 터널)

PC 없이 Colab 하나에서 다음을 모두 합니다.

1. Colab GPU에 Qwen 모델을 **직접 로드** (`chatbot/backends/local_backend.py`, transformers)
2. 같은 런타임에서 솔챗 **웹 서버**(FastAPI) 실행
3. Cloudflare 임시 터널로 **공개 주소**를 만들어 PC·휴대폰 브라우저에서 접속

`serve_vllm_colab.ipynb` 와의 차이: 그쪽은 모델만 Colab에 띄우고 웹 서버는 PC에서 돌립니다.
이 노트북은 웹 서버까지 Colab에서 돌립니다.

주의
- Colab 세션이 살아 있는 동안만 동작합니다. 탭을 닫거나 오래 두면 끊기고, 주소와 대화 기록이 사라집니다.
- Colab 약관상 계속 열어두는 공개 서비스 용도는 맞지 않습니다. 혼자 확인하는 데모용으로 쓰세요.
- `configs/model.yaml` 원본은 고치지 않고, Colab 전용 설정 파일을 따로 만들어 씁니다.

**시작 전**
- `런타임 > 런타임 유형 변경 > T4 GPU`
- 왼쪽 🔑 (보안 비밀)에 `ACCESS_CODE`(웹 접속 비밀번호)를 추가하고 노트북 액세스를 켜세요.
  Gemini도 같이 쓰고 싶으면 `OPENROUTER_API_KEY` 도 추가합니다 (없어도 됩니다).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/sol_chat'   # 구글 드라이브의 sol_chat 폴더
%cd {ROOT}

In [ ]:
# 서비스 + 로컬 모델 로드에 필요한 패키지 (torch 는 Colab 에 이미 있음)
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# 설정: 여기 값만 바꾸면 됩니다
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # Hugging Face 모델 ID
ADAPTER = ''                                # 학습한 LoRA 어댑터 경로 (예: 'outputs/adapters/sol-lora-v1'), 없으면 빈칸
PORT = 8000

In [ ]:
# 비밀값을 🔑 보안 비밀에서 읽어 환경 변수로 넘김 (.env 를 Drive 에 둘 필요 없음)
import os, secrets
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

env = dict(os.environ)
env['ACCESS_CODE'] = secret('ACCESS_CODE')
if not env['ACCESS_CODE']:
    env['ACCESS_CODE'] = secrets.token_urlsafe(9)
    print('ACCESS_CODE 보안 비밀이 없어서 임시 비밀번호를 만들었어요:', env['ACCESS_CODE'])
if secret('OPENROUTER_API_KEY'):
    env['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY')
    print('OPENROUTER_API_KEY 확인: Gemini 모델도 선택할 수 있어요.')

In [ ]:
# Colab 전용 모델 설정 만들기: 원래 model.yaml 에 로컬 모델 항목을 추가하고 기본값으로 지정
import yaml

with open('configs/model.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

entry = {'label': f'{BASE_MODEL.split("/")[-1]} (Colab GPU)', 'backend': 'local', 'base_model': BASE_MODEL}
if ADAPTER:
    entry['adapter_path'] = ADAPTER
    entry['label'] = '내 LoRA 모델 (Colab GPU)'
cfg['models'] = {'colab-local': entry, **cfg['models']}
cfg['default_model'] = 'colab-local'

MODEL_CONFIG = '/content/sol_chat_model.yaml'
with open(MODEL_CONFIG, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
print('사용할 모델 목록:', list(cfg['models']))

In [ ]:
# 웹 서버를 백그라운드로 실행
import subprocess, textwrap

launcher = textwrap.dedent(f'''
    import sys
    sys.path.insert(0, {ROOT!r})
    import uvicorn
    from chatbot.config import load_chat_config
    from server.main import create_app

    app = create_app(chat_config=load_chat_config({MODEL_CONFIG!r}))
    uvicorn.run(app, host="127.0.0.1", port={PORT})
''')
with open('/content/run_sol_chat.py', 'w') as f:
    f.write(launcher)

server = subprocess.Popen(['python', '/content/run_sol_chat.py'], cwd=ROOT, env=env,
                          stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)

In [ ]:
# 서버가 뜰 때까지 대기
import time, requests
for _ in range(60):
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/', timeout=3).ok:
            print('웹 서버 준비 완료')
            break
    except requests.ConnectionError:
        pass
    time.sleep(2)
else:
    print('서버가 뜨지 않았어요. 아래 로그를 확인하세요.')
    print(open('/content/server.log').read()[-3000:])

In [ ]:
# 모델 미리 불러오기: 첫 질문 때 로드하면 오래 걸리므로 여기서 한 번 호출 (모델 다운로드 포함 수 분)
headers = {'X-Access-Code': env['ACCESS_CODE'], 'X-Client-Id': 'colab-warmup'}
r = requests.post(f'http://127.0.0.1:{PORT}/api/chat', headers=headers,
                  json={'message': '안녕, 너는 누구야?'}, timeout=900)
print(r.status_code, r.json())

In [ ]:
# Cloudflare 임시 터널로 공개 주소 만들기
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
URL = None
for _ in range(30):
    time.sleep(2)
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if found:
        URL = found.group(0)
        break
print('접속 주소 :', URL)
print('비밀번호  :', env['ACCESS_CODE'])

위 주소를 PC나 휴대폰 브라우저에서 열고 비밀번호를 입력하면 됩니다.
웹 화면 왼쪽의 모델 선택에서 Colab 모델과 (키를 넣었다면) Gemini를 오가며 비교할 수 있어요.

In [ ]:
# (문제가 생겼을 때) 서버 로그 마지막 부분 보기
print(open('/content/server.log').read()[-3000:])

In [ ]:
# (끝낼 때) 서버와 터널 종료
tunnel.terminate(); server.terminate()
print('종료했어요.')